[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/tusharislampure29/sanskrit-ocr-correction/blob/main/notebooks/train_colab.ipynb) &nbsp;
[![Open in Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://kaggle.com/kernels/welcome?src=https://github.com/tusharislampure29/sanskrit-ocr-correction/blob/main/notebooks/train_colab.ipynb)

# Sanskrit Post-OCR Correction — ByT5 fine-tuning (Colab T4)

Trains a byte-level **ByT5-small** to correct OCR'd Sanskrit/Devanagari text, using
a linguistically-grounded synthetic corruption engine. Designed for a **free Colab T4**.

**Before you Run all:**
1. Runtime → Change runtime type → **T4 GPU**.
2. Add your Hugging Face token as a Colab **Secret** named `HF_TOKEN` (🔑 in the left bar),
   with write access. (Used to push checkpoints — so a Colab disconnect costs minutes, not the run.)

Repo: https://github.com/tusharislampure29/sanskrit-ocr-correction  ·  Model → `tusharislampure29/byt5-sanskrit-ocr`  ·  Dataset → `tusharislampure29/sanskrit-ocr-correction`

In [ ]:
# 1. GPU check + deps. Colab already ships a matched torch/torchvision pair —
#    install ONLY the missing pieces (upgrading torch breaks the CUDA pairing).
import subprocess, torch
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
# Pin transformers to a known-good 4.x: transformers 5.x removed
# Seq2SeqTrainingArguments(overwrite_output_dir=...) and changed Trainer internals.
subprocess.run("pip install -q 'transformers==4.46.3' 'datasets>=2.20' 'accelerate>=0.34' "
               "sentencepiece evaluate huggingface_hub", shell=True, check=True)
import transformers; print("transformers", transformers.__version__)

In [ ]:
# 2. Auth to Hugging Face from the Colab secret.
from huggingface_hub import login
from google.colab import userdata
login(userdata.get('HF_TOKEN'))
HF_MODEL_ID   = "tusharislampure29/byt5-sanskrit-ocr"
HF_DATASET_ID = "tusharislampure29/sanskrit-ocr-correction"

In [ ]:
# 3. Clone the repo (single source of truth for src/ + the curated corpus).
import os
if not os.path.exists('sanskrit-ocr-correction'):
    subprocess.run("git clone https://github.com/tusharislampure29/sanskrit-ocr-correction", shell=True, check=True)
else:
    subprocess.run("git -C sanskrit-ocr-correction pull", shell=True, check=True)
os.chdir('sanskrit-ocr-correction')
print(os.getcwd())

In [ ]:
# 4. Build the dataset on Colab: curated corpus + Ayurveda pages + HF augmentation.
#    Sanskrit Wikipedia (Parquet, no loading script -> reliable on modern datasets).
#    ~3k clean lines x 10 variants ≈ 24-30k corrupted->clean pairs. (Sized so the
#    fp32 T4 run + per-epoch generative eval finish in a couple of hours.)
subprocess.run('python -m src.data_prep --hf-dataset "wikimedia/wikipedia:20231101.sa" '
               '--max-hf 3000 --max-clean 3000 --variants 10', shell=True, check=True)
import json
print(open('data/processed/stats.json', encoding='utf-8').read())

In [ ]:
# 5. Push the dataset to the Hub (so it's a citable artifact + reproducible).
from datasets import load_dataset
ds = load_dataset('json', data_files={
    'train': 'data/processed/train.jsonl',
    'validation': 'data/processed/val.jsonl',
    'test': 'data/processed/test.jsonl'})
print(ds)
try:
    ds.push_to_hub(HF_DATASET_ID, private=False)
    print('dataset pushed ->', HF_DATASET_ID)
except Exception as e:
    print('dataset push skipped:', e)

In [ ]:
# 6. Tokenize. ByT5 is byte-level: no OOV on Devanagari, corruption-invariant vocab.
from transformers import AutoTokenizer
BASE = 'google/byt5-small'
PREFIX = 'correct: '
MAX_SRC = MAX_TGT = 384
tok = AutoTokenizer.from_pretrained(BASE)

def preprocess(batch):
    model_in = tok([PREFIX + x for x in batch['noisy']], max_length=MAX_SRC,
                   truncation=True)
    labels = tok(text_target=batch['clean'], max_length=MAX_TGT, truncation=True)
    model_in['labels'] = labels['input_ids']
    return model_in

tokenized = ds.map(preprocess, batched=True, remove_columns=ds['train'].column_names)
print(tokenized)

In [ ]:
# 7. Metric: character error rate (CER), the right metric for OCR correction.
import numpy as np
def _lev(a, b):
    if a == b: return 0
    if not a: return len(b)
    if not b: return len(a)
    prev = list(range(len(b)+1))
    for i, ca in enumerate(a, 1):
        cur = [i]
        for j, cb in enumerate(b, 1):
            cur.append(min(prev[j]+1, cur[-1]+1, prev[j-1]+(ca!=cb)))
        prev = cur
    return prev[-1]

def cer(pred, ref):
    return _lev(pred, ref)/max(1, len(ref))

def compute_metrics(eval_pred):
    preds, labels = eval_pred
    if isinstance(preds, tuple): preds = preds[0]
    preds = np.where(preds != -100, preds, tok.pad_token_id)
    labels = np.where(labels != -100, labels, tok.pad_token_id)
    dpred = tok.batch_decode(preds, skip_special_tokens=True)
    dref  = tok.batch_decode(labels, skip_special_tokens=True)
    return {'cer': float(np.mean([cer(p, r) for p, r in zip(dpred, dref)]))}

In [ ]:
# 8. Train. push_to_hub + save each epoch => resume-safe across disconnects.
#
# *** CRITICAL: precision. ***
# T5/ByT5 are numerically UNSTABLE in fp16 — activations overflow fp16's range,
# the loss goes to NaN on step 1, and you get a dead model (this exact bug bit an
# earlier run: val_loss=nan, CER stuck at 1.08). So:
#   - bf16 if the GPU supports it (A100 / RTX 3090+/4090/5090) — stable AND fast,
#   - else fp32 on a T4 (Turing has no bf16 hardware; software bf16 is too slow).
# NEVER fp16 here.
import torch, transformers
from transformers import (AutoModelForSeq2SeqLM, DataCollatorForSeq2Seq,
                          Seq2SeqTrainingArguments, Seq2SeqTrainer)
model = AutoModelForSeq2SeqLM.from_pretrained(BASE)
collator = DataCollatorForSeq2Seq(tok, model=model)

use_bf16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()
print(f"precision -> {'bf16' if use_bf16 else 'fp32'} (fp16 is disabled: unsafe for T5/ByT5)")

training_args = Seq2SeqTrainingArguments(
    output_dir='byt5-sanskrit-ocr',
    learning_rate=3e-4, per_device_train_batch_size=4,
    per_device_eval_batch_size=8, gradient_accumulation_steps=4,
    num_train_epochs=3, warmup_ratio=0.05, weight_decay=0.01,
    bf16=use_bf16, fp16=False,   # fp16 NEVER for T5/ByT5 (see note above)
    predict_with_generate=True, generation_max_length=MAX_TGT,
    logging_steps=50, eval_strategy='epoch', save_strategy='epoch', save_total_limit=2,
    load_best_model_at_end=True, metric_for_best_model='cer', greater_is_better=False,
    push_to_hub=True, hub_model_id=HF_MODEL_ID, hub_strategy='every_save',
    report_to='none')

# Cap the per-epoch eval set: generative eval (predict_with_generate) is slow,
# and a 400-sample CER is a perfectly good early signal that training is healthy.
_eval_ds = tokenized['validation']
if len(_eval_ds) > 400:
    _eval_ds = _eval_ds.select(range(400))

trainer = Seq2SeqTrainer(
    model=model, args=training_args,
    train_dataset=tokenized['train'], eval_dataset=_eval_ds,
    data_collator=collator, compute_metrics=compute_metrics)
trainer.train()
trainer.push_to_hub()
tok.push_to_hub(HF_MODEL_ID)
print('model pushed ->', HF_MODEL_ID)

In [ ]:
# 9. Evaluate on the held-out TEST set: before (noisy) vs after (corrected).
# Baseline first (no GPU, can't fail); then the model eval. Neither hard-fails the
# notebook (check=False) — the model is already safe on the Hub by this point, and
# the eval can also be re-run locally on CPU via --load-responses.
import gc, json, torch
# Free the T4's VRAM left over from training before spawning the eval process,
# otherwise the still-resident trainer/model can push the eval into CUDA OOM
# (the most common cause of a red error on this cell).
for _name in ['trainer', 'model', 'collator']:
    globals().pop(_name, None)
gc.collect()
torch.cuda.empty_cache()
print('--- BASELINE (uncorrected noisy vs clean) ---')
subprocess.run("python -m src.eval_harness --baseline --test data/processed/test.jsonl", shell=True)
print('--- MODEL (corrected vs clean) ---')
r = subprocess.run(f"python -m src.eval_harness --model {HF_MODEL_ID} "
                   f"--test data/processed/test.jsonl --save eval/results/preds_test.json",
                   shell=True)
if r.returncode != 0:
    print('NOTE: model eval exited non-zero (often CUDA OOM on a busy T4). The model is '
          'already on the Hub; re-run this cell, or score on CPU with '
          '`python -m src.eval_harness --load-responses eval/results/preds_test.json`.')

In [ ]:
# 10. Controlled error-taxonomy: what does the model actually fix?
subprocess.run(f"python -m src.eval_harness --taxonomy --model {HF_MODEL_ID} "
               f"--test data/processed/test.jsonl", shell=True)

In [ ]:
# 11. Qualitative demo on the Ayurveda pages ImmverseAI shipped.
subprocess.run(f"python -m src.inference --model {HF_MODEL_ID} --demo-pages", shell=True)

In [ ]:
# 12. Charts (training loss + before/after CER) and download eval artifacts.
subprocess.run("python scripts/make_charts.py", shell=True)
from google.colab import files
for f in ['eval/results/preds_test.json', 'eval/results/taxonomy.json',
          'eval/charts/eval_comparison.png']:
    try: files.download(f)
    except Exception as e: print('skip', f, e)